# Hardship scoring: interpretable household need score

This notebook creates an interpretable 0-to-100 hardship score from raw synthetic indicators. The goal is not to replace professional judgment. The goal is to show how transparent scoring can support triage, dashboarding, and program planning.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("synthetic_food_housing_insecurity_100k.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/synthetic_food_housing_insecurity_100k.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Build normalized risk components

Each component is scaled from 0 to 1, where higher means greater hardship. The weights are explicit so they can be reviewed and changed.

In [ ]:
def clip_scale(series, low, high, reverse=False):
    """Scale a numeric series to 0-1 after clipping to low/high."""
    scaled = (series.clip(low, high) - low) / (high - low)
    if reverse:
        scaled = 1 - scaled
    return scaled.fillna(0)

risk = pd.DataFrame(index=df.index)

risk["income_pressure"] = clip_scale(df["income_to_poverty_ratio"], 0.5, 3.0, reverse=True)
risk["housing_cost_pressure"] = clip_scale(df["rent_or_mortgage_burden_pct"], 20, 70)
risk["housing_instability"] = (
    0.45 * df["behind_on_housing_payment"]
    + 0.35 * df["eviction_or_foreclosure_risk"]
    + 0.20 * df["utility_shutoff_notice"]
)
risk["financial_fragility"] = (
    0.50 * clip_scale(df["savings_buffer_days"], 0, 60, reverse=True)
    + 0.30 * clip_scale(df["debt_to_income_pct"], 0, 80)
    + 0.20 * df["income_volatility"].map({"low": 0.0, "medium": 0.5, "high": 1.0}).fillna(0.5)
)
risk["employment_instability"] = df["employment_status"].map({
    "employed_full_time": 0.0,
    "employed_part_time": 0.35,
    "gig_or_contract": 0.45,
    "unemployed": 1.0,
    "not_in_labor_force": 0.65,
    "retired": 0.25,
}).fillna(0.4)
risk["food_access_pressure"] = (
    0.40 * clip_scale(df["distance_to_grocery_miles"], 0, 15)
    + 0.35 * df["food_desert_flag"]
    + 0.25 * (df["transportation_access"].ne("reliable")).astype(int)
)
risk["digital_and_service_access"] = (
    0.45 * (df["internet_access"].ne("broadband")).astype(int)
    + 0.35 * df["childcare_access_barrier"].map({"not_applicable": 0, "none": 0, "minor": 0.35, "major": 1.0}).fillna(0)
    + 0.20 * (df["health_insurance_status"].ne("insured")).astype(int)
)
risk["health_family_need"] = (
    0.35 * df["disability_present"]
    + 0.25 * df["chronic_health_condition"]
    + 0.20 * df["children_present"]
    + 0.20 * df["senior_present"]
)
risk["stress_signal"] = clip_scale(df["mental_health_stress_score"], 0, 12)

weights = pd.Series({
    "income_pressure": 0.18,
    "housing_cost_pressure": 0.12,
    "housing_instability": 0.16,
    "financial_fragility": 0.15,
    "employment_instability": 0.08,
    "food_access_pressure": 0.10,
    "digital_and_service_access": 0.08,
    "health_family_need": 0.07,
    "stress_signal": 0.06,
})

assert abs(weights.sum() - 1.0) < 1e-9
risk.head()

## 2. Calculate the score and bands

Bands convert the numeric score into plain-language levels for dashboarding and triage.

In [ ]:
scored = df.copy()
scored["interpretable_hardship_score"] = (risk * weights).sum(axis=1).mul(100).round(1)

scored["hardship_band"] = pd.cut(
    scored["interpretable_hardship_score"],
    bins=[-0.1, 25, 50, 75, 100],
    labels=["low", "moderate", "high", "severe"],
)

scored[["household_id", "interpretable_hardship_score", "hardship_band", "food_insecure_label", "housing_insecure_label"]].head()

## 3. Validate the score against synthetic outcomes

A useful score should show higher food and housing insecurity rates in higher score bands. This is not causal validation; it is a face-validity check.

In [ ]:
band_summary = (
    scored.groupby("hardship_band", observed=False)
    .agg(
        households=("household_id", "count"),
        avg_score=("interpretable_hardship_score", "mean"),
        food_insecure_rate=("food_insecure_label", "mean"),
        housing_insecure_rate=("housing_insecure_label", "mean"),
        avg_original_hardship_score=("overall_hardship_score", "mean"),
    )
    .reset_index()
)

correlation = scored["interpretable_hardship_score"].corr(scored["overall_hardship_score"])
print("Correlation with existing synthetic hardship score:", round(correlation, 3))
band_summary.round(3)

## 4. Decile lift chart

The chart shows whether the highest-scored deciles contain a larger share of food-insecure and housing-insecure households.

In [ ]:
scored["hardship_decile"] = pd.qcut(scored["interpretable_hardship_score"], 10, labels=False, duplicates="drop") + 1

deciles = (
    scored.groupby("hardship_decile")
    .agg(
        households=("household_id", "count"),
        avg_score=("interpretable_hardship_score", "mean"),
        food_insecure_rate=("food_insecure_label", "mean"),
        housing_insecure_rate=("housing_insecure_label", "mean"),
    )
    .reset_index()
)

deciles.round(3)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(deciles["hardship_decile"], deciles["food_insecure_rate"], marker="o", label="Food insecure")
plt.plot(deciles["hardship_decile"], deciles["housing_insecure_rate"], marker="o", label="Housing insecure")
plt.xlabel("Hardship score decile")
plt.ylabel("Rate")
plt.title("Synthetic outcome rates by hardship decile")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Save scored data

The scored file can be used for dashboarding, triage lists, and policy-priority modeling.

In [ ]:
out_cols = [
    "household_id",
    "interpretable_hardship_score",
    "hardship_band",
    "food_insecure_label",
    "housing_insecure_label",
    "policy_priority_segment",
]
scored[out_cols].to_csv("household_hardship_scores.csv", index=False)
Path("household_hardship_scores.csv")